# Plotting Basics - IRIS-HEP Analysis Training

Authored by: [Andrzej Novak](https://github.com/andrzejnovak), [Matthew Feickert](https://github.com/matthewfeickert), [Roger Janusiak](https://github.com/RogerJanusiak)

This notebook is the reference text for the module. Read and run it top to bottom: Ch. 0 sets up the tools, Ch. 1 covers `matplotlib`, Ch. 2 `mplhep`, and Ch. 3 `hist`. Each chapter ends with a recap and the problems in `hist_mplhep_problem_set.ipynb` that go with it.

## Ch. 0 - Setup and Introduction

### 0.1 - Background Info

#### Histograms mean different things in different contexts
- Counts, bin edges &mdash; useful for a bar plot &mdash; `np.histogram` / `plt.bar`
- Counts, bin edges, pre computed errors &mdash; `TGraphErrors`/`plt.errorbar`
- Weighted values, weights squared, bin_edges &mdash; proper error calculation `TH1`/`Coffea.hist`/`hist`

#### UHI - [Unified Histogram Interface](https://uhi.readthedocs.io/en/latest/plotting.html#using-the-protocol)
- (Plottable) Histogram protocol designed to make libraries interoperable, easy to navigate
  - Conformed to by `hist`, `mplhep`, `uproot`, `histoprint`, and now also ROOT (full UHI support in ROOT coming soon)!
- Each UHI histogram has the following methods
  - `h.values()`: The value (as given by the kind)
  - `h.variances()`: The variance in the value (None if an unweighed histogram was filled with weights)
  - `h.counts()`: How many fills the bin received or the effective number of fills if the histogram is weighted
  - `h.axes`: A Sequence of axes
  - and a few other properties

#### [hist](https://github.com/scikit-hep/hist)
* Python go to one-stop for histogramming
* Extends [boost-histogram](https://github.com/scikit-hep/boost-histogram) (Python binding for C++ `Boost::Histogram` library &mdash; *FAST*)
  - Makes it user friendly
* Shortcuts for convenience and interactive plotting/fitting

#### [mplhep](https://github.com/scikit-hep/mplhep)
- Built on top of `matplotlib`
- Extends functionality to easily plot histograms from various inputs
- Holds style sheets for easy experiment specific style application

### 0.2 - Setup

This module uses `mplhep`, `hist`, [`scikit-hep-testdata`](https://github.com/scikit-hep/scikit-hep-testdata), `uproot`, and &mdash; for one section and one problem &mdash; ROOT. Install them with whichever environment manager you use:

```console
# pip / virtual environment
$ python -m pip install --upgrade mplhep hist uproot scikit-hep-testdata

# conda, including Coffea-casa
$ conda upgrade --yes mplhep hist
$ conda install --channel conda-forge --yes uproot scikit-hep-testdata root_base

# pixi
$ pixi upgrade hist
$ pixi add mplhep uproot scikit-hep-testdata root_base
```

ROOT (`root_base`) is optional &mdash; skip it and skip Section 2.3's PyROOT cell.

#### Tip

You can run any of these from a notebook by prefixing the line with `!`, which escapes out to the shell:

```
! conda upgrade --yes mplhep hist
```

## Ch. 1 - `matplotlib`

### 1.1 - Tail of Two APIs of matplotlib

**References**

* [Anatomy of a [Matplotlib] figure](https://matplotlib.org/stable/gallery/showcase/anatomy.html)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

In [ ]:
fig_dir = Path.cwd() / "figures"
fig_dir.mkdir(exist_ok=True)

Matplotlib has two [high level APIs](https://matplotlib.org/stable/api/index.html):
* The `pyplot` interface (function-based, implicit)
* The `Axes` interface (object-based, explicit)

#### Stateful (`pyplot` API)

If we execute a command using the `pyplot` API we see that a global state object is created

In [ ]:
plt.plot(np.arange(0, 10, 1), np.linspace(0, 1, 10))

We can continue to operate on this global state `Axes` through the `plt` API even though we aren't giving a specific `matplotlib.Axes` object to operate on

In [ ]:
plt.plot(np.arange(0, 10, 1), np.linspace(0, 1, 10))
plt.title("pyplot Example")
plt.savefig(fig_dir / "pyplot_example.png")

#### Object-oriented (`Axes` API)

With the `Axes` API, we need to explicitly create a [`Figure`](https://matplotlib.org/stable/api/_as_gen/matplotlib.figure.Figure.html#matplotlib.figure.Figure) and [`Axes`](https://matplotlib.org/stable/api/_as_gen/matplotlib.axes.Axes.html#matplotlib.axes.Axes) object and then call functions that operate explicitily on the `Axes` object.

In [ ]:
fig, ax = plt.subplots()

In [ ]:
ax.plot(np.linspace(0, 1, 10), np.linspace(0, 10, 10))
ax.set_title("Axes Example")

In [ ]:
fig.savefig(fig_dir / "axes_example.png")

fig

You might notice that we still used the `pyplot` API (`plt.subplots`) even though we're using the `Axes` API. That's a shortcut &mdash; you can also use the full object oriented Matplotlib API:

In [ ]:
from matplotlib.figure import Figure

fig = Figure()
ax = fig.subplots()

x = np.linspace(0, 10, 1000)
y = np.sin(x)

ax.plot(x, y)
fig.savefig(fig_dir / "full_axes_api_example.png")

fig

**Rule of thumb:** `pyplot` is fine for a quick look at an array; use the `Axes` API for anything that ends up in a script, a paper, or a multi-panel figure. Every `mplhep` plotting function takes an `ax=` argument for exactly this reason.

### 1.2 - Histogramming in matplotlib

The simplest way to draw pre-binned counts is [`ax.bar`](https://matplotlib.org/stable/api/_as_gen/matplotlib.axes.Axes.bar.html), which takes bin *centers* and heights. (Don't confuse it with `ax.hist`, which takes raw, unbinned data and does the binning for you.) While we are here: `set_xlabel`/`set_ylabel` are how you label the axes &mdash; a plot without them is not ready to show anyone.

In [ ]:
counts = [1, 2, 3, 4, 2, 1, 0]
centers = np.arange(len(counts)) + 0.5

fig, ax = plt.subplots()
ax.bar(centers, counts, width=1.0)
ax.set_xlabel("x")
ax.set_ylabel("Counts")
ax.set_title("ax.bar");

Bars are not how we usually draw histograms in HEP though. Through the [`stairs` API](https://matplotlib.org/stable/api/_as_gen/matplotlib.axes.Axes.stairs.html) (contributed from HEP) we can arrive at something that looks familiar to us. `stairs` also takes bin *contents*, not raw data.

In [ ]:
fig, ax = plt.subplots()
ax.stairs([1, 2, 3, 4, 2, 1, 0])

Pass the bin edges as the second argument &mdash; there is always one more edge than there are bin contents. `fill=True` with `baseline=0` gives the filled look.

In [ ]:
edges = np.arange(len(counts) + 1)  # one more edge than bin contents

fig, ax = plt.subplots()
ax.stairs(counts, edges, baseline=0, fill=True)
ax.set_xlabel("x")
ax.set_ylabel("Counts");

Histograms can be drawn on top of each other

In [ ]:
a, b = [1, 2, 3, 4, 2, 1, 0], [1, 2, 3, 2, 2, 3, 1]

fig, ax = plt.subplots()
ax.stairs(a, label="A")
ax.stairs(b, label="B", ls="--")
ax.legend()

`stairs` understands how to operate on [Array API](https://proceedings.scipy.org/articles/gerudo-f2bc6f59-001) compatible objects (like NumPy arrays), and `baseline` lets you stack by hand

In [ ]:
fig, ax = plt.subplots()
ax.stairs(np.sum([a, b], axis=0), baseline=b, fill=True, label="A")
ax.stairs(b, fill=True, label="B")
ax.legend()

### 1.3 - Recap

* `pyplot` is implicit and stateful, the `Axes` API is explicit and reusable &mdash; prefer the latter for real plots.
* `ax.stairs` draws pre-binned contents as a step histogram, optionally filled, and stacked through `baseline`.
* Doing this by hand gets tedious quickly: no uncertainties, no stacking, no labels. That is what Ch. 2 fixes.

**Practice:** problems 1.1&ndash;1.3 of `hist_mplhep_problem_set.ipynb`.

## Ch. 2 - `mplhep`

### 2.1 - Setup and first plot

`mplhep` gives us a high level API that allows for quickly getting the information that we traditionally think about when it comes to histograms in particle physics.

In [ ]:
import mplhep

In [ ]:
yields, bins = np.histogram(np.random.normal(5, 1, 5000), bins=10)

In [ ]:
# bins are optional, they default to unit-width integer bins
mplhep.histplot(yields)

The docstring is the fastest reference for everything `histplot` accepts

In [ ]:
? mplhep.histplot

### 2.2 - Using the API

#### Primary goal is to stay unobtrusive, if it works in `matplotlib`, it should work in `mplhep`

Pass `ax=` to draw into a specific subplot, bin edges as the second argument, and `yerr=True` for uncertainties.

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(12, 4))

mplhep.histplot(yields, ax=axs[0])
mplhep.histplot(yields, bins, yerr=True, ax=axs[1])
axs[1].set_title("Uncertainties shown");

#### Using kwargs in `mplhep`

`kwargs` are passed though to `matplotlib`, so styling works exactly as you already know it. `histtype` picks the representation: `"step"` (default), `"fill"`, or `"errorbar"`.

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(12, 4))

mplhep.histplot(
    yields, ax=axs[0], histtype="fill", hatch="///", edgecolor="red", facecolor="none"
)
axs[0].set_title("Filled histogram")
mplhep.histplot(
    yields, ax=axs[1], histtype="errorbar", yerr=True, color="black", capsize=4
)
axs[1].set_title("Uncertainity outline");

#### Stacking and normalizing is available

Pass a list of histograms to draw several at once: `stack=True` piles them up, `density=True` normalizes to unit area for shape comparisons, and `label=` feeds the legend.

In [ ]:
bin_contents = yields.copy()
fig, axs = plt.subplots(1, 3, figsize=(18, 4))

data = np.random.poisson(bin_contents * 3)
mplhep.histplot(
    [bin_contents, bin_contents * 2],
    bins=bins,
    ax=axs[0],
    yerr=True,
    label=["MC1", "MC2"],
)
mplhep.histplot(data, bins=bins, ax=axs[1], yerr=True, label="Data")

mplhep.histplot(
    [bin_contents, bin_contents * 2],
    bins=bins,
    ax=axs[2],
    stack=True,
    label=["MC1", "MC2"],
    density=True,
)
mplhep.histplot(
    data,
    bins=bins,
    ax=axs[2],
    yerr=True,
    histtype="errorbar",
    label="Data",
    density=True,
    color="k",
)
for ax in axs:
    ax.legend()
axs[0].set_title("Some MCs")
axs[1].set_title("Draw Poisson Data")
axs[2].set_title("Data/MC Shape comparison");

#### Convenient sorting options

`sort="yield"` orders a stack by integral and `sort="label"` alphabetically; add the `_r` suffix to reverse either.

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(12, 4))
mplhep.histplot(
    [bin_contents * 2, bin_contents * 3, bin_contents],
    bins=bins,
    ax=axs[0],
    stack=True,
    histtype="fill",
    label=["A", "B", "C"],
    sort="yield",
)
mplhep.histplot(
    [bin_contents * 2, bin_contents * 3, bin_contents],
    bins=bins,
    ax=axs[1],
    stack=True,
    histtype="fill",
    label=["A", "B", "C"],
    sort="label_r",
)
for ax in axs:
    ax.legend()
axs[0].set_title("Sort by yield")
axs[1].set_title("Sort by label (add _r to reverse)");

### 2.3 - Histograms from other libraries

So far we have handed `histplot` NumPy arrays, but its real convenience is that **any object following the UHI protocol can be passed in directly** &mdash; no unpacking of contents, edges, and errors on your side.

#### From a ROOT file with `uproot`

`uproot.open` takes a path to a ROOT file. If you don't have one handy, `scikit-hep-testdata` ships sample files: `from skhep_testdata import data_path` then `data_path("uproot-hepdata-example.root")` returns a local path to one.

In [ ]:
import uproot

root_file = uproot.open("some_file.root")
print(root_file.keys())

uproot_hist = root_file["my_hist"]
mplhep.histplot(uproot_hist, yerr=True, label="From uproot")
plt.legend();

#### From PyROOT

Recent ROOT releases implement UHI too, so a `TH1` also goes straight into `histplot`. (Skip this cell if you did not install ROOT.)

In [ ]:
import ROOT

th1 = ROOT.TH1F("h1", "h1", 50, -2.5, 2.5)
th1.FillRandom("gaus", 10000)

mplhep.histplot(th1, yerr=True, label="From PyROOT")
plt.legend();

If you want to *manipulate* the histogram (slice, rebin, project &mdash; see Ch. 3) rather than only draw it, convert it to a `hist.Hist` first:

```python
uproot_hist.to_hist()                     # uproot object -> hist.Hist
uproot.pyroot.from_pyroot(th1).to_hist()  # PyROOT TH1    -> hist.Hist
```

### 2.4 - Styling with mplhep

* Primary purpose of `mplhep` is to serve and distribute styles
   - **ALICE**
   - **ATLAS**
   - **CMS**
   - **LHCb**
* To ensure plots looks the same on any framework fonts need to be included
   - I am liable to go on a rant, so suffice to say:
      - We package an open look-alike of Helvetica called Tex Gyre Heros

A style is just a bundle of `rcParams`, so `mplhep.style.use` also takes a list where later entries override earlier ones.

In [ ]:
mplhep.style.use([mplhep.style.ATLAS, {"figure.figsize": (8, 8)}])
mplhep.histplot(np.histogram(np.random.normal(10, 3, 1000)), histtype="fill")
mplhep.atlas.label();

#### Label variants

`mplhep.<experiment>.label()` writes the standardized label block. The arguments you will reach for most:

* first argument (`text=`) &mdash; the qualifier: `"Internal"`, `"Preliminary"`, `"Work in Progress"`, ...
* `data=` &mdash; `False` (the default) prepends *Simulation*
* `lumi=`, `year=`, `com=` &mdash; right-hand side information (luminosity in fb$^{-1}$, year, centre-of-mass energy in TeV)
* `loc=` &mdash; layout, from `0` (above the axes, the default) to `4` (ATLAS-style block inside the axes, which shows energy and luminosity but not the year)
* `ax=` &mdash; which subplot to label

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(14, 5))

mplhep.histplot(np.histogram(np.random.normal(10, 3, 1000)), histtype="fill", ax=axs[0])
mplhep.atlas.label("Internal", data=True, lumi=140, com=13.6, loc=4, ax=axs[0])

mplhep.histplot(np.histogram(np.random.normal(10, 3, 1000)), histtype="fill", ax=axs[1])
mplhep.atlas.label("Preliminary", data=False, lumi=140, year=2018, ax=axs[1]);

Styles are global: `mplhep.style.use(mplhep.style.CMS)` (or `ALICE`, `LHCb`) switches experiment, and `plt.style.use("default")` gets you back to stock matplotlib.

### 2.5 - Color recommendations

- Data should be always shown in black. Basic color recommendations with examples are found below.

- Categorical Data (e.g. 1D Stackplots): Use the color sequence suggested by M. Petroff in [Accessible Color Sequences for Data Visualization](https://arxiv.org/abs/2107.02270) and [available on GitHub](https://github.com/mpetroff/accessible-color-cycles) (MIT License).
- Specifically you should use the Petroff 6-color cycle:
`["#5790fc", "#f89c20", "#e42536", "#964a8b", "#9c9ca1", "#7a21dd"]`

In [ ]:
from matplotlib.colors import ListedColormap

petroff6 = ListedColormap(
    ["#5790fc", "#f89c20", "#e42536", "#964a8b", "#9c9ca1", "#7a21dd"]
)
petroff6

 - or if more colors are needed the Petroff 10-color cycle:
    ```
    ["#3f90da", "#ffa90e", "#bd1f01", "#94a4a2", "#832db6", "#a96b59", "#e76300", "#b9ac70", "#717581", "#92dadd"]
    ```

which was added to Matplotlib in `v3.10.0`.

In [ ]:
# matplotlib v3.10.0+
plt.style.use("petroff10")

# on matplotlib v3.9.x build it by hand instead:
# petroff10 = ListedColormap(["#3f90da", "#ffa90e", "#bd1f01", "#94a4a2", "#832db6",
#                             "#a96b59", "#e76300", "#b9ac70", "#717581", "#92dadd"])

In [ ]:
import matplotlib as mpl

ListedColormap(mpl.color_sequences["petroff10"])

### 2.6 - Recap

* `mplhep.histplot(contents, bins, yerr=True, ax=...)` is the workhorse; kwargs it doesn't use fall through to `matplotlib`.
* `histtype=` picks step/fill/errorbar; `stack=`, `density=`, and `sort=` handle multi-histogram plots.
* Any UHI object &mdash; `uproot`, PyROOT, `hist` &mdash; can be passed straight in.
* `mplhep.style.use(...)` plus `mplhep.<experiment>.label(...)` gives publication-ready styling; use the Petroff cycles for categories and black for data.

**Practice:** problems 2.1&ndash;2.3 of `hist_mplhep_problem_set.ipynb`.

## Ch. 3 - `hist`

### 3.1 - First Histogram with `hist`

NumPy arrays stop being convenient as soon as a histogram has more than one axis, carries weights, or needs to be sliced by category. A `hist.Hist` is a set of **axes** plus a **storage**.

In [ ]:
import hist

The most explicit way to build one is to name every piece:

In [ ]:
# histogram creation
one_axis_hist = hist.Hist(
    hist.axis.Regular(10, 0, 10, name="x", label="x-axis"), hist.storage.Int64()
)

one_axis_hist.fill(x=[1, 1, 4, 6])

one_axis_hist

Displaying a histogram gives a summary of its axes and storage plus, for 1D, an ASCII preview. `name=` is how you refer to an axis when slicing; `label=` is what gets drawn on the plot.

### 3.2 - Quick hist creation

Instead of having to define each axis as its own `hist.axis` object, you can also create the same histogram by chaining axes together. This is useful for quickly creating or redefining histograms.

In [ ]:
# histogram creation
h = (
    hist.new.Regular(10, 0, 10, name="x", label="x-axis")
    .Variable(range(10), name="y", label="y-axis")
    .Int64()
    .fill(*np.random.multivariate_normal([4, 6], [[2, 0], [0, 1]], 10000).T)
)

h

In [ ]:
# even quicker - every axis type has a short alias
h = hist.new.Reg(10, 0, 10).Var(range(10)).Int64()
h

### 3.3 - Axis types

`hist` allows for [multiple kinds of axes](https://hist.readthedocs.io/en/latest/user-guide/axes.html#axis-types) from [`boost-histogram`](https://github.com/scikit-hep/boost-histogram):

* [Regular](https://hist.readthedocs.io/en/latest/user-guide/axes.html#regular-axis)
* Boolean
* [Variable](https://hist.readthedocs.io/en/latest/user-guide/axes.html#variable-axis) (variable width bins)
* Integer
* [IntCategory](https://hist.readthedocs.io/en/latest/user-guide/axes.html#category-axis) (bins correspond to categories that are indexed by integer values, e.g. `[2, 5, 7, 3, 9]`)
* [StrCategory](https://hist.readthedocs.io/en/latest/user-guide/axes.html#category-axis) (bins correspond to categories that are indexed by string values, e.g. `["Electron", "Muon"]`)

In [ ]:
axis0 = hist.axis.Regular(10, -5, 5, overflow=False, underflow=False, name="A")
axis1 = hist.axis.Boolean(name="B")
axis2 = hist.axis.Variable(range(10), name="C")
axis3 = hist.axis.Integer(-5, 5, overflow=False, underflow=False, name="D")
axis4 = hist.axis.IntCategory(range(10), name="E")
axis5 = hist.axis.StrCategory(["Electron", "Muon"], name="F", label="Particles")

Categorical axes can also **grow**: start one empty with `growth=True` and a bin appears whenever an unseen category is filled. This is how you accumulate one histogram per sample or per systematic without knowing the list up front.

In [ ]:
# Growth! A new category bin appears whenever an unseen category is filled
h = hist.new.Reg(10, 0, 10, name="x").StrCat([], growth=True, name="dataset").Weight()
h.fill(x=np.random.normal(5, 2, 1000), dataset="A")
h.fill(x=np.random.normal(7, 2, 1000), dataset="B")

h

### 3.4 - Storage types

A number of possible [storage types](https://hist.readthedocs.io/en/latest/user-guide/storages.html) exist: `Double`, `Unlimited`, `Int64`, `AtomicInt64`, `Weight`, `Mean`, and `WeightedMean`.

In practice you will most commonly use `Weight()`, which keeps both the sum of weights and the sum of squared weights &mdash; the second is what keeps uncertainties correct.

By default, the `weight`s will be `1`

In [ ]:
hist.new.Reg(10, 0, 10).Weight().fill([1, 2, 3, 5]).plot();

and you can pass a `weight` for each filled value

In [ ]:
hist.new.Reg(10, 0, 10).Weight().fill([1, 2, 3, 5], weight=[1, 1, 1, 0.5]).plot();

### 3.5 - Hist manipulation and UHI

This is where the multidimensional object pays off. Every operation below returns a *new* histogram, so they chain freely and never modify the original.

For more information on `hist` check out the user guide: https://hist.readthedocs.io/

and for `uhi` the docs live at https://uhi.readthedocs.io/

In [ ]:
# example histogram
example_hist = (
    hist.new.Reg(10, 0, 10, name="x")
    .Var(range(10), name="y")
    .Var(range(10), name="z")
    .Weight()
    .fill(*np.random.multivariate_normal([4, 6, 4], np.eye(3), 100000).T)
)

example_hist

In [ ]:
# Project on an axis: keeps the named axes, sums over the rest
example_hist.project("x")

In [ ]:
example_hist.project("x", "y")

In [ ]:
# Slicing (applying cuts). Indices are bin numbers by default and `sum` integrates an
# axis away - here so that we are left with something 2D to draw.
example_hist[5:, :, sum].plot();

By default if we index a histogram array we are indexing by _bin_. We can also use the [`j` suffix syntax to index by _value_](https://uhi.readthedocs.io/en/latest/indexing%2B.html)

In [ ]:
# Indexing by bin 5 onwards for x, and by value 6 onwards for y
example_hist[5:, 6j:, sum].plot();

Dictionary access does the same by axis *name*, which beats counting commas

In [ ]:
example_hist[5:, :, sum][{"y": 5}].plot();

and the two styles compose, so complex selections stay one-liners

In [ ]:
example_hist[5:, :, sum][{"y": 6, "x": sum}]
# example_hist[5:, :, sum][{"y": 6, "x": 7j}]

Inside a dictionary you cannot write `5:` directly, so `hist` provides a `Slicer` that turns `s[...]` into a slice

In [ ]:
# Makes slicing inside dictionaries simpler
s = hist.tag.Slicer()

# ... and combined with hist.rebin it covers merging bins
example_hist[{"x": s[2j:8j], "y": s[:: hist.rebin(2)], "z": s[::sum]}]

#### UHI cheat sheet

| Operation | Positional | By axis name |
| --- | --- | --- |
| Bins 5 onwards | `h[5:, :, :]` | `h[{"x": s[5:]}]` |
| From value 6 onwards | `h[6j:, :, :]` | `h[{"x": s[6j:]}]` |
| Integrate an axis away | `h[:, :, sum]` | `h[{"z": s[::sum]}]` |
| Cut, *then* integrate | &mdash; | `h[{"z": s[6j::sum]}]` |
| Keep only some axes | `h.project("x", "y")` | &mdash; |
| Merge groups of 2 bins | &mdash; | `h[{"x": s[:: hist.rebin(2)]}]` |
| Pick one category | &mdash; | `h[{"dataset": "A"}]` |

The "cut, then integrate" row is the workhorse in a real analysis: `s[0.5j::sum]` on an MVA axis means *keep score > 0.5 and sum what is left*.

### 3.6 - Plotting `hist` objects

`hist` objects know how to draw themselves: `.plot()` dispatches to `.plot1d()`/`.plot2d()` and is itself built on `mplhep`. For full control hand the histogram to `mplhep.histplot` instead &mdash; because `hist` follows UHI, contents, edges, and uncertainties all come along.

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(12, 4))

example_hist.project("x").plot(ax=axs[0])
mplhep.histplot(
    example_hist.project("x"), yerr=True, histtype="errorbar", color="k", ax=axs[1]
)
axs[0].set_title(".plot()")
axs[1].set_title("mplhep.histplot()");

A categorical axis becomes a set of histograms with `.stack()`, the idiomatic way to draw one entry per sample

In [ ]:
h.stack("dataset").plot(stack=True, histtype="fill")
plt.legend();

And when you need the numbers themselves &mdash; for a fit, a ratio, or a figure of merit &mdash; the UHI accessors hand them back as arrays:

```python
h.values()               # bin contents
h.variances()            # squared uncertainties
h.axes["x"].centers      # bin centers
h.axes["x"].edges        # bin edges
list(h.axes["dataset"])  # category names
```

### 3.7 - Recap

* A `hist.Hist` is axes + storage: build it explicitly with `hist.Hist(...)` or fluently with `hist.new...`.
* Regular/Boolean/Variable/Integer/IntCategory/StrCategory cover the analysis cases; `growth=True` lets categories appear as they are filled.
* `Weight()` storage is what keeps uncertainties correct.
* UHI gives you `project`, bin and value slicing, `sum`, `rebin`, and category selection &mdash; all non-destructive.
* `h.plot()` for a quick look, `mplhep.histplot(h, ...)` when you want control.

**Practice:** problems 3.1&ndash;3.2 and the Final Exam of `hist_mplhep_problem_set.ipynb`.